In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
matplotlib.use('Agg')

print("All imports good — starting Project 1: Cell Anomaly Detector")

All imports good — starting Project 1: Cell Anomaly Detector


In [3]:
np.random.seed(42)

# Number of samples per cell state
n_samples = 500

def generate_healthy(n):
    """Normal cell — all KPIs in expected range"""
    return pd.DataFrame({
        'dl_sinr':          np.random.normal(15, 2, n),      # good SINR 13-17dB
        'cqi_mean':         np.random.normal(10, 1, n),      # CQI 9-11, good channel
        'cqi_std':          np.random.normal(1.5, 0.3, n),   # stable CQI
        'mcs_mean':         np.random.normal(20, 2, n),      # high MCS, good throughput
        'mcs_low_pct':      np.random.normal(5, 2, n),       # few low MCS allocations
        'prb_util_dl':      np.random.normal(40, 10, n),     # moderate PRB usage
        'pdcch_util':       np.random.normal(30, 8, n),      # moderate PDCCH
        'pucch_util':       np.random.normal(25, 7, n),      # moderate PUCCH
        'dl_throughput_mbps': np.random.normal(45, 8, n),   # healthy throughput
        'label': 'healthy'
    })

def generate_coverage_issue(n):
    """Poor coverage — SINR low, CQI low, MCS low, scheduler compensates with PRB"""
    return pd.DataFrame({
        'dl_sinr':          np.random.normal(3, 2, n),       # poor SINR <5dB
        'cqi_mean':         np.random.normal(4, 1, n),       # low CQI, bad channel
        'cqi_std':          np.random.normal(2.5, 0.5, n),   # unstable CQI
        'mcs_mean':         np.random.normal(8, 2, n),       # low MCS forced
        'mcs_low_pct':      np.random.normal(60, 10, n),     # majority low MCS
        'prb_util_dl':      np.random.normal(75, 10, n),     # high PRB — scheduler
                                                              # allocating more RBs
                                                              # to compensate
        'pdcch_util':       np.random.normal(65, 8, n),      # high control channel
        'pucch_util':       np.random.normal(55, 8, n),      # high uplink control
        'dl_throughput_mbps': np.random.normal(8, 3, n),    # throughput collapsed
        'label': 'coverage'
    })

def generate_congestion(n):
    """Congestion — SINR fine, CQI fine, MCS fine, PRB maxed out"""
    return pd.DataFrame({
        'dl_sinr':          np.random.normal(14, 2, n),      # SINR still good
        'cqi_mean':         np.random.normal(9, 1, n),       # CQI still reasonable
        'cqi_std':          np.random.normal(1.8, 0.3, n),   # fairly stable
        'mcs_mean':         np.random.normal(18, 2, n),      # MCS still decent
        'mcs_low_pct':      np.random.normal(10, 3, n),      # few low MCS
        'prb_util_dl':      np.random.normal(95, 3, n),      # PRB maxed — cell full
        'pdcch_util':       np.random.normal(88, 5, n),      # PDCCH nearly full
        'pucch_util':       np.random.normal(80, 6, n),      # PUCCH under pressure
        'dl_throughput_mbps': np.random.normal(15, 5, n),   # throughput limited
                                                             # by resources not radio
        'label': 'congestion'
    })

# Combine all three states into one dataset
df = pd.concat([
    generate_healthy(n_samples),
    generate_coverage_issue(n_samples),
    generate_congestion(n_samples)
], ignore_index=True)

# Clip unrealistic values
df['dl_sinr'] = df['dl_sinr'].clip(-10, 30)
df['cqi_mean'] = df['cqi_mean'].clip(1, 15)
df['mcs_mean'] = df['mcs_mean'].clip(0, 28)
df['mcs_low_pct'] = df['mcs_low_pct'].clip(0, 100)
df['prb_util_dl'] = df['prb_util_dl'].clip(0, 100)
df['pdcch_util'] = df['pdcch_util'].clip(0, 100)
df['pucch_util'] = df['pucch_util'].clip(0, 100)
df['dl_throughput_mbps'] = df['dl_throughput_mbps'].clip(0, 150)

print(f"Dataset shape: {df.shape}")
print(f"\nSamples per class:")
print(df['label'].value_counts())
print(f"\nKPI ranges by cell state:")
print(df.groupby('label')[['dl_sinr', 'cqi_mean', 'mcs_mean', 
                             'prb_util_dl', 'dl_throughput_mbps']].mean().round(1))

Dataset shape: (1500, 10)

Samples per class:
label
healthy       500
coverage      500
congestion    500
Name: count, dtype: int64

KPI ranges by cell state:
            dl_sinr  cqi_mean  mcs_mean  prb_util_dl  dl_throughput_mbps
label                                                                   
congestion     14.0       9.0      17.8         95.0                15.4
coverage        3.0       4.0       8.0         75.0                 8.1
healthy        15.0      10.0      20.1         40.2                44.2


In [4]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('KPI Distribution by Cell State', fontsize=14, fontweight='bold')

kpis_to_plot = [
    ('dl_sinr', 'DL SINR (dB)'),
    ('cqi_mean', 'CQI Mean'),
    ('mcs_mean', 'MCS Mean'),
    ('mcs_low_pct', 'Low MCS %'),
    ('prb_util_dl', 'PRB Utilization DL %'),
    ('pdcch_util', 'PDCCH Utilization %'),
    ('pucch_util', 'PUCCH Utilization %'),
    ('dl_throughput_mbps', 'DL Throughput (Mbps)'),
    ('cqi_std', 'CQI Std Dev'),
]

colors = {
    'healthy': 'steelblue',
    'coverage': 'coral',
    'congestion': 'goldenrod'
}

for idx, (kpi, label) in enumerate(kpis_to_plot):
    ax = axes[idx // 3][idx % 3]
    
    for state in ['healthy', 'coverage', 'congestion']:
        subset = df[df['label'] == state][kpi]
        ax.hist(subset, bins=30, alpha=0.6, 
                color=colors[state], label=state)
    
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('kpi_distributions.png', dpi=150, bbox_inches='tight')
plt.close()

print("Saved: kpi_distributions.png")
print("\nOpen the file and tell me what you see.")
print("Key things to look for:")
print("  SINR — do the three states separate cleanly?")
print("  PRB  — does congestion sit clearly above healthy?")
print("  Throughput — do coverage and congestion both drop but for different reasons?")

Saved: kpi_distributions.png

Open the file and tell me what you see.
Key things to look for:
  SINR — do the three states separate cleanly?
  PRB  — does congestion sit clearly above healthy?
  Throughput — do coverage and congestion both drop but for different reasons?


In [5]:
# Step 1 — Prepare features and labels
# Drop the label column to get features only
X = df.drop(columns=['label'])
y = df['label']

print(f"Features: {list(X.columns)}")
print(f"Dataset shape: {X.shape}")

# Step 2 — Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

# Step 3 — Train the classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
print("\nModel trained.")

# Step 4 — Evaluate
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nOverall accuracy: {accuracy*100:.1f}%")

# Step 5 — Per class accuracy
print("\nAccuracy per cell state:")
for state in ['healthy', 'coverage', 'congestion']:
    mask = y_test == state
    state_acc = accuracy_score(y_test[mask], y_pred[mask])
    print(f"  {state}: {state_acc*100:.1f}%")

# Step 6 — Feature importance
# Which KPI does the model rely on most?
importances = pd.Series(
    clf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nFeature importance (which KPIs matter most):")
for kpi, importance in importances.items():
    bar = '█' * int(importance * 100)
    print(f"  {kpi:<20} {importance:.3f} {bar}")

Features: ['dl_sinr', 'cqi_mean', 'cqi_std', 'mcs_mean', 'mcs_low_pct', 'prb_util_dl', 'pdcch_util', 'pucch_util', 'dl_throughput_mbps']
Dataset shape: (1500, 9)

Training samples: 1200
Test samples: 300

Model trained.

Overall accuracy: 100.0%

Accuracy per cell state:
  healthy: 100.0%
  coverage: 100.0%
  congestion: 100.0%

Feature importance (which KPIs matter most):
  dl_sinr              0.160 ███████████████
  mcs_low_pct          0.155 ███████████████
  pdcch_util           0.154 ███████████████
  cqi_mean             0.133 █████████████
  pucch_util           0.132 █████████████
  prb_util_dl          0.129 ████████████
  dl_throughput_mbps   0.074 ███████
  mcs_mean             0.064 ██████
  cqi_std              0.000 
